# BankScope retrieval evaluacija u Google Colab-u

Ovaj notebook pokreće kompletan retrieval evaluator nad postojećim BankScope
artefaktima:

```text
Qwen dense + BM25S -> RRF hybrid -> Qwen3-Reranker-0.6B
```

Notebook ne generiše ponovo corpus embeddinge. Učitava validirani NPZ sa
7.110 zapisa, proverava njegov integritet, izvršava evaluator na GPU-u i čuva
rezultate nazad na Google Drive.

Pre pokretanja izaberi `Runtime -> Change runtime type -> T4 GPU`, pa zatim
pokreni `Runtime -> Run all`.


## Goal

Dobiti reproduktivan A/B rezultat za metode `dense`, `bm25`, `hybrid` i
`reranked`, bez menjanja projekta ili ponovnog embedding run-a.

Potrebni Drive fajlovi:

- `MyDrive/BankScope/data/processed/embedding_records/sec_10k_embedding_records.jsonl`
- `MyDrive/BankScope/data/processed/embeddings/qwen3_embedding_0_6b_records.npz`

`retrieval_queries_dev.jsonl` dolazi iz GitHub repozitorijuma. BM25 indeks se
gradi u memoriji i nema poseban artefakt na Drive-u.


## Setup

### 1. Parameters


In [43]:
from pathlib import Path

REPOSITORY_URL = (
    "https://github.com/nikolabakic/"
    "Banking-Technology-and-Operational-Risk-Intelligence-Assistant.git"
)
REPOSITORY_ROOT = Path(
    "/content/Banking-Technology-and-Operational-Risk-Intelligence-Assistant"
)
DRIVE_ROOT = Path("/content/drive/MyDrive/BankScope")

EXPECTED_RECORD_COUNT = 7_110
EXPECTED_EMBEDDING_DIMENSION = 1_024

CANDIDATE_K = 30
RRF_K = 60
RERANKER_BATCH_SIZE = 4


### 2. Verify the GPU


In [44]:
import torch

assert torch.cuda.is_available(), (
    "CUDA nije dostupna. U Colab-u izaberi "
    "Runtime -> Change runtime type -> T4 GPU."
)

print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch CUDA:", torch.version.cuda)


GPU: Tesla T4
PyTorch CUDA: 12.8


### 3. Mount Drive and optionally load an HF token

Qwen modeli su javni, pa Hugging Face token nije obavezan. Ako želiš veće
rate limite, dodaj novi secret pod nazivom `HF_TOKEN` u Colab `Secrets`.
Token se nikada ne upisuje direktno u notebook.


In [45]:
import os

from google.colab import drive, userdata

drive.mount("/content/drive")

try:
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    print("HF token je učitan iz Colab Secrets.")
else:
    print("HF token nije podešen. Javni Qwen modeli će i dalje raditi.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
HF token nije podešen. Javni Qwen modeli će i dalje raditi.


### 4. Clone or update the repository


In [46]:
import subprocess

if (REPOSITORY_ROOT / ".git").exists():
    subprocess.run(
        ["git", "-C", str(REPOSITORY_ROOT), "pull", "--ff-only"],
        check=True,
    )
else:
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            REPOSITORY_URL,
            str(REPOSITORY_ROOT),
        ],
        check=True,
    )

commit_hash = subprocess.check_output(
    ["git", "-C", str(REPOSITORY_ROOT), "rev-parse", "HEAD"],
    text=True,
).strip()

print("Repository:", REPOSITORY_ROOT)
print("Commit:", commit_hash)


Repository: /content/Banking-Technology-and-Operational-Risk-Intelligence-Assistant
Commit: 83654a777dc5eb102a5dd8939051b8c38712073a


### 5. Install retrieval dependencies

Colab trenutno koristi Python 3.12, dok ceo projekat u `pyproject.toml` traži
Python 3.13. Za evaluaciju ne menjamo projektni requirement i ne koristimo
`pip install -e .`. Instaliramo samo potrebne retrieval biblioteke i eksplicitno
dodajemo `src` u Python putanju.


In [47]:
import importlib
import sys

retrieval_dependencies = [
    "bm25s>=0.3,<0.4",
    "sentence-transformers>=5.0,<6.0",
    "transformers>=4.51,<6.0",
]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", *retrieval_dependencies],
    check=True,
)

source_path = str(REPOSITORY_ROOT / "src")
if source_path not in sys.path:
    sys.path.insert(0, source_path)

os.environ["PYTHONPATH"] = os.pathsep.join(
    path
    for path in [source_path, os.environ.get("PYTHONPATH", "")]
    if path
)
importlib.invalidate_caches()

import bankscope

print("Python:", sys.version.split()[0])
print("BankScope:", bankscope.__file__)


Python: 3.12.13
BankScope: /content/Banking-Technology-and-Operational-Risk-Intelligence-Assistant/src/bankscope/__init__.py


## Steps

### 6. Copy the two large artifacts from Drive


In [55]:
import shutil

artifact_pairs = {
    DRIVE_ROOT
    / "data/processed/embedding_records/sec_10k_embedding_records.jsonl": (
        REPOSITORY_ROOT
        / "data/processed/embedding_records/sec_10k_embedding_records.jsonl"
    ),
    DRIVE_ROOT
    / "data/processed/embeddings/qwen3_embedding_0_6b_records.npz": (
        REPOSITORY_ROOT
        / "data/processed/embeddings/qwen3_embedding_0_6b_records.npz"
    ),
}

for source_file, destination_file in artifact_pairs.items():
    if not source_file.exists():
        raise FileNotFoundError(
            f"Nedostaje Drive fajl: {source_file}. "
            "Proveri strukturu MyDrive/BankScope/data."
        )

    destination_file.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source_file, destination_file)
    size_mb = destination_file.stat().st_size / (1024**2)
    print(f"Copied: {destination_file.relative_to(REPOSITORY_ROOT)} ({size_mb:.2f} MB)")

queries_path = REPOSITORY_ROOT / "data/evaluation/retrieval_queries_dev.jsonl"
assert queries_path.exists(), f"Nedostaje query fajl iz repozitorijuma: {queries_path}"
print("Queries:", queries_path.relative_to(REPOSITORY_ROOT))


Copied: data/processed/embedding_records/sec_10k_embedding_records.jsonl (20.97 MB)
Copied: data/processed/embeddings/qwen3_embedding_0_6b_records.npz (29.70 MB)
Queries: data/evaluation/retrieval_queries_dev.jsonl


## Checks

### 7. Validate NPZ and JSONL alignment


In [56]:
import json

import numpy as np

records_path = (
    REPOSITORY_ROOT
    / "data/processed/embedding_records/sec_10k_embedding_records.jsonl"
)
embeddings_path = (
    REPOSITORY_ROOT
    / "data/processed/embeddings/qwen3_embedding_0_6b_records.npz"
)

with records_path.open(encoding="utf-8") as file:
    records = [json.loads(line) for line in file if line.strip()]

with np.load(embeddings_path, allow_pickle=False) as archive:
    embeddings = np.asarray(archive["embeddings"])
    npz_record_ids = archive["record_ids"].astype(str).tolist()
    embedding_model_name = str(archive["model_name"].item())

jsonl_record_ids = [str(record["record_id"]) for record in records]
embedding_norms = np.linalg.norm(embeddings, axis=1)

assert len(records) == EXPECTED_RECORD_COUNT
assert embeddings.shape == (
    EXPECTED_RECORD_COUNT,
    EXPECTED_EMBEDDING_DIMENSION,
)
assert embeddings.dtype == np.float32
assert np.isfinite(embeddings).all()
assert np.all(embedding_norms > 0)
assert np.allclose(embedding_norms, 1.0, atol=1e-4)
assert npz_record_ids == jsonl_record_ids

print("Model:", embedding_model_name)
print("Records:", len(records))
print("Embedding shape:", embeddings.shape)
print("Embedding dtype:", embeddings.dtype)
print(
    "Norm range:",
    f"{embedding_norms.min():.8f} - {embedding_norms.max():.8f}",
)
print("NPZ and JSONL record order: OK")


Model: Qwen/Qwen3-Embedding-0.6B
Records: 7110
Embedding shape: (7110, 1024)
Embedding dtype: float32
Norm range: 0.99999988 - 1.00000012
NPZ and JSONL record order: OK


### 8. Validate the evaluator version and query set


In [57]:
evaluator_path = REPOSITORY_ROOT / "scripts/evaluate_retrieval.py"
evaluator_source = evaluator_path.read_text(encoding="utf-8")

assert 'METHODS = ("dense", "bm25", "hybrid", "reranked")' in evaluator_source
assert "reranker_model_name" in evaluator_source
assert "get_hybrid_candidates" in evaluator_source

with queries_path.open(encoding="utf-8") as file:
    queries = [json.loads(line) for line in file if line.strip()]

answerable_queries = [query for query in queries if query.get("status") == "answerable"]

assert answerable_queries, "Query skup nema nijedno answerable pitanje."

print("Evaluator supports: dense, bm25, hybrid, reranked")
print("All queries:", len(queries))
print("Answerable queries:", len(answerable_queries))

if len(answerable_queries) < 30:
    print(
        "Napomena: ovo je mali development/smoke skup. "
        "Ne koristi ga za konačnu procenu kvaliteta sistema."
    )


Evaluator supports: dense, bm25, hybrid, reranked
All queries: 2
Answerable queries: 2
Napomena: ovo je mali development/smoke skup. Ne koristi ga za konačnu procenu kvaliteta sistema.


### 9. Run the full evaluator

Prvo pokretanje preuzima `Qwen3-Reranker-0.6B` i može nekoliko minuta izgledati
kao da miruje. Model se zatim čuva u Colab kešu za ostatak runtime-a.


In [51]:
evaluation_command = [
    sys.executable,
    str(evaluator_path),
    "--candidate-k",
    str(CANDIDATE_K),
    "--rrf-k",
    str(RRF_K),
    "--reranker-device",
    "cuda",
    "--reranker-batch-size",
    str(RERANKER_BATCH_SIZE),
]

evaluation_environment = os.environ.copy()
evaluation_environment["PYTHONPATH"] = os.environ["PYTHONPATH"]

print("Running:", " ".join(evaluation_command))
subprocess.run(
    evaluation_command,
    cwd=REPOSITORY_ROOT,
    env=evaluation_environment,
    check=True,
)


Running: /usr/bin/python3 /content/Banking-Technology-and-Operational-Risk-Intelligence-Assistant/scripts/evaluate_retrieval.py --candidate-k 30 --rrf-k 60 --reranker-device cuda --reranker-batch-size 4


CompletedProcess(args=['/usr/bin/python3', '/content/Banking-Technology-and-Operational-Risk-Intelligence-Assistant/scripts/evaluate_retrieval.py', '--candidate-k', '30', '--rrf-k', '60', '--reranker-device', 'cuda', '--reranker-batch-size', '4'], returncode=0)

### 10. Inspect and verify the metrics


In [52]:
import pandas as pd
from IPython.display import display

results_path = (
    REPOSITORY_ROOT / "data/evaluation/results/retrieval_eval_dev.json"
)
misses_path = (
    REPOSITORY_ROOT
    / "data/evaluation/results/retrieval_top1_misses_dev.jsonl"
)

evaluation = json.loads(results_path.read_text(encoding="utf-8"))
methods = list(evaluation["overall"])

assert methods == ["dense", "bm25", "hybrid", "reranked"]
assert evaluation["reranker_model_name"] == "Qwen/Qwen3-Reranker-0.6B"
assert evaluation["record_count"] == EXPECTED_RECORD_COUNT

summary_rows = []
for method, metrics in evaluation["overall"].items():
    summary_rows.append(
        {
            "method": method,
            "queries": metrics["query_count"],
            "hit_at_1": metrics["hit_rate_at_1"],
            "hit_at_3": metrics["hit_rate_at_3"],
            "hit_at_5": metrics["hit_rate_at_5"],
            "hit_at_10": metrics["hit_rate_at_10"],
            "mean_recall_at_5": metrics["mean_recall_at_5"],
            "mrr_at_10": metrics["mrr_at_10"],
        }
    )

summary = pd.DataFrame(summary_rows).set_index("method")

print("Query file:", evaluation["query_file"])
print("Methods:", methods)
display(summary.round(4))


Query file: /content/Banking-Technology-and-Operational-Risk-Intelligence-Assistant/data/evaluation/retrieval_queries_dev.jsonl
Methods: ['dense', 'bm25', 'hybrid', 'reranked']


,queries,hit_at_1,hit_at_3,hit_at_5,hit_at_10,mean_recall_at_5,mrr_at_10
method,,,,,,,
dense,2,0.0,0.5,0.5,1.0,0.20,0.3125
bm25,2,0.5,0.5,0.5,0.5,0.10,0.5000
hybrid,2,0.0,1.0,1.0,1.0,0.45,0.5000
reranked,2,0.5,1.0,1.0,1.0,0.45,0.6667


### 11. Inspect top-1 misses


In [53]:
with misses_path.open(encoding="utf-8") as file:
    top1_misses = [json.loads(line) for line in file if line.strip()]

miss_rows = []
for miss in top1_misses:
    first_result = miss["retrieved"][0] if miss["retrieved"] else {}
    miss_rows.append(
        {
            "query_id": miss["query_id"],
            "method": miss["method"],
            "ticker": miss.get("ticker"),
            "question": miss["query"],
            "top_1_target_chunk_id": first_result.get("target_chunk_id"),
            "top_1_preview": first_result.get("preview", "")[:180],
        }
    )

miss_table = pd.DataFrame(miss_rows)
print("Top-1 misses:", len(miss_table))
display(miss_table.head(20))


Top-1 misses: 6


,query_id,method,ticker,question,top_1_target_chunk_id,top_1_preview
0,dev_jpm_standardized_cet1_ratio_2025,dense,JPM,What was JPMorgan Chase & Co.'s Standardized C...,653d8ffbdc0646ee96d37ea57afb761f7d4ac1a01711e4...,"December 31, 2024 (in millions, except ratios)..."
1,dev_jpm_standardized_cet1_ratio_2025,hybrid,JPM,What was JPMorgan Chase & Co.'s Standardized C...,653d8ffbdc0646ee96d37ea57afb761f7d4ac1a01711e4...,"December 31, 2024 (in millions, except ratios)..."
2,dev_jpm_operational_risk_causes_2025,dense,JPM,What causes operational risk according to JPMo...,8f106940a9c6145cab20451d0dfc5a2b259b3dc9503570...,• errors made by JPMorganChase or another mark...
3,dev_jpm_operational_risk_causes_2025,bm25,JPM,What causes operational risk according to JPMo...,f334a5bd335e6e7257153c70c9eb9e4dbdb6f228f3e174...,"long-term debt issuances, and its intermediate..."
4,dev_jpm_operational_risk_causes_2025,hybrid,JPM,What causes operational risk according to JPMo...,7baf47111d4fb6a53e4461f18cf6d93770d16c60e8e54f...,"Part I • actions by banking regulators, as wel..."
5,dev_jpm_operational_risk_causes_2025,reranked,JPM,What causes operational risk according to JPMo...,8f106940a9c6145cab20451d0dfc5a2b259b3dc9503570...,• errors made by JPMorganChase or another mark...


### 12. Save results to Drive


In [54]:
drive_results_directory = DRIVE_ROOT / "data/evaluation/results"
drive_results_directory.mkdir(parents=True, exist_ok=True)

for result_file in (results_path, misses_path):
    destination = drive_results_directory / result_file.name
    shutil.copy2(result_file, destination)
    print("Saved:", destination)


Saved: /content/drive/MyDrive/BankScope/data/evaluation/results/retrieval_eval_dev.json
Saved: /content/drive/MyDrive/BankScope/data/evaluation/results/retrieval_top1_misses_dev.jsonl


## Next Steps

Pošalji sledeća dva fajla iz `MyDrive/BankScope/data/evaluation/results`:

- `retrieval_eval_dev.json`
- `retrieval_top1_misses_dev.jsonl`

Pri tumačenju prvo proveravamo da li `reranked` popravlja `Hit@1` i `MRR@10`
u odnosu na `hybrid`, a zatim ručno pregledamo top-1 promašaje. Ako je skup i
dalje sastavljen od samo dva answerable pitanja, rezultat ostaje integracioni
smoke test. Sledeći ozbiljan korak je proširenje na najmanje 30 ručno proverenih
development pitanja.
